# Sorpresas inflacionarias y volatilidad del S&P Merval (Argentina, 2021-2025) — v2

**Etapa de datos.** Este notebook arma la base de eventos que después estima el Rmd (`Sorpresas_Merval_v2.Rmd`). Reemplaza al notebook anterior (`sorpresas_merval_documentado.ipynb`), que bajaba todo de Yahoo Finance.

**Qué cambia respecto de la versión anterior** (bitácora del grupo, ítems 34-38, y pedido de la cátedra):

1. **Precios de Bloomberg** (Merval, 7 acciones, GGAL local y ADR para el CCL), descargados por el grupo. El VIX viene de Yahoo Finance (la terminal no lo permitió).
2. **Sorpresa contra el REM del mismo mes**: la última encuesta relevada antes de que INDEC publique el dato (10 a 18 días antes), en vez de la de fines del mes previo (41 a 49 días antes). La medida anterior se conserva como robustez.
3. **Retorno anormal con modelo de media constante** (MacKinlay, 1997), estimado en una ventana previa a cada evento.
4. **CCL implícito con GGAL** (BYMA/NYSE, ratio 10) en lugar de YPF.
5. **Variables de contexto**: EMBI medido 5 ruedas antes del evento y episodios de crisis cambiaria definidos con una regla objetiva (salto diario del dólar oficial BNA mayor al 10%).

La salida son tres CSV que lee el Rmd: `base_eventos_v2.csv`, `merval_diario_v2.csv` y `tabla_yahoo_vs_bloomberg.csv`.

In [18]:
!pip install yfinance openpyxl --quiet

## Parte 1 — Configuración

Subí a una carpeta de Drive (o a la sesión de Colab) estos archivos, sin cambiarles el nombre:

- la carpeta `datos bloomberg/` completa (Merval, 7 acciones, `CCL_GGAL.xlsx`, `historico_vix.xlsx`);
- `Dolar BNA.xlsx` y `EMBI 2020-2026.xlsx` (carpeta "DATOS COMPLETOS" del grupo);
- `historico-relevamiento-expectativas-mercado.xlsx` (REM del BCRA).

In [19]:
import os, datetime as dt
import numpy as np, pandas as pd, openpyxl

# Si se trabaja desde Drive:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

RUTA = os.environ.get("RUTA_DATOS", "datos/")   # <-- ajustar
RUTA_BBG = RUTA + "datos bloomberg/"
TICKERS = ["GGAL", "BMA", "YPFD", "PAMP", "CEPU", "TXAR", "ALUA"]
FIN_MUESTRA = "2026-02-27"     # cubre la ventana del último evento (IPC dic-2025, publicado el 13-ene-2026)

## Parte 2 — Series de Bloomberg y control de calidad

Los Excel de Bloomberg traen dos columnas (`Date`, `PX_LAST`) con filas vacías al principio. La función toma solo las filas con fecha y valor numérico.

In [20]:
def leer_dos_columnas(path, col=1):
    ws = openpyxl.load_workbook(path, data_only=True, read_only=True).worksheets[0]
    filas = []
    for r in ws.iter_rows(values_only=True):
        if r and isinstance(r[0], dt.datetime) and len(r) > col and r[col] not in (None, ""):
            try:
                filas.append((pd.Timestamp(r[0]), float(str(r[col]).replace(",", ""))))
            except ValueError:
                pass
    s = pd.DataFrame(filas, columns=["fecha", "valor"]).drop_duplicates("fecha").sort_values("fecha")
    return s.set_index("fecha")["valor"]

series = {"merval": leer_dos_columnas(RUTA_BBG + "Merval cierre.xlsx"),
          "vix": leer_dos_columnas(RUTA_BBG + "historico_vix.xlsx")}
for t in TICKERS:
    series[t] = leer_dos_columnas(RUTA_BBG + f"{t}-ARS-COMPLETO.xlsx")

control = pd.DataFrame([(k, len(s), s.index.min().date(), s.index.max().date(), (s.index.weekday >= 5).sum(),
                         (s <= 0).sum(), (s.diff() == 0).sum()) for k, s in series.items()],
                       columns=["serie", "n", "desde", "hasta", "fin_de_semana", "no_positivos", "valor_repetido"])
control

,serie,n,desde,hasta,fin_de_semana,no_positivos,valor_repetido
0,merval,1222,2020-12-30,2025-12-30,0,0,2
1,vix,1726,2019-12-30,2026-11-06,0,0,8
2,GGAL,1673,2019-12-30,2026-11-06,0,0,23
3,BMA,1673,2019-12-30,2026-11-06,0,0,15
4,YPFD,1673,2019-12-30,2026-11-06,0,0,7
5,PAMP,1673,2019-12-30,2026-11-06,0,0,17
6,CEPU,1673,2019-12-30,2026-11-06,0,0,31
7,TXAR,1673,2019-12-30,2026-11-06,0,0,40
8,ALUA,1673,2019-12-30,2026-11-06,0,0,40


### 2.1 CCL implícito con GGAL

El archivo `CCL_GGAL.xlsx` del grupo ya trae GGAL en BYMA y su ADR en NYSE (ratio 10 acciones por ADR). Cuando uno de los dos mercados no opera, arrastra el último cierre y lo marca en la columna `Arrastrado`.

In [21]:
ws = openpyxl.load_workbook(RUTA_BBG + "CCL_GGAL.xlsx", data_only=True, read_only=True).worksheets[0]
filas = [r[:7] for r in ws.iter_rows(values_only=True) if r and isinstance(r[0], dt.datetime)]
ccl_tab = pd.DataFrame(filas, columns=["fecha", "ars", "usd", "ars_u", "usd_u", "ccl", "arrastrado"])
ccl_tab["fecha"] = pd.to_datetime(ccl_tab["fecha"])
ccl = ccl_tab.set_index("fecha")["ccl"].astype(float)

tramo = ccl_tab[(ccl_tab.fecha >= "2021-01-01") & (ccl_tab.fecha <= FIN_MUESTRA)]
print("Días 2021 - fin de muestra:", len(tramo))
print("Días con precio arrastrado:", tramo["arrastrado"].fillna("ninguno").value_counts().to_dict())

Días 2021 - fin de muestra: 1328
Días con precio arrastrado: {'ninguno': 1226, 'ARS': 68, 'USD': 34}


### 2.2 Merval: Bloomberg contra Yahoo Finance

El Merval de Bloomberg termina el 30-dic-2025. Se compara contra Yahoo (`^MERV`) para documentar que el cambio de fuente no altera los datos y para justificar dos correcciones puntuales:

- **13-oct-2025**: Bloomberg repite el cierre del 9-oct, pero ese día hubo rueda (el feriado se trasladó al 10-oct). Se usa el cierre de Yahoo.
- **31-dic-2025 a 27-feb-2026**: no está en la descarga de Bloomberg y hace falta para la ventana del IPC de diciembre 2025 (publicado el 13-ene-2026). Se completa con Yahoo.

In [22]:
try:
    import yfinance as yf
    yahoo = yf.Ticker("^MERV").history(start="2020-11-01", end="2026-03-31", auto_adjust=False)["Close"]
    if isinstance(yahoo, pd.DataFrame):
        yahoo = yahoo.iloc[:, 0]
    yahoo.index = pd.to_datetime(yahoo.index).tz_localize(None)
    if yahoo.dropna().empty:
        raise ValueError("yfinance devolvió una serie vacía")
except Exception as e:
    print("yfinance no disponible, se usa el CSV de la versión anterior:", e)
    yahoo = pd.read_csv(RUTA + "merval_diario_con_volatilidad.csv", parse_dates=["fecha"]).set_index("fecha")["merval_close"]
yahoo = yahoo.dropna().rename("yahoo")

bbg = series["merval"].rename("bloomberg")
comp = pd.concat([bbg, yahoo], axis=1, sort=True).loc["2020-12-30":"2025-12-30"]
ambos = comp.dropna()
dif = ambos.bloomberg / ambos.yahoo - 1
ret_b, ret_y = np.log(ambos.bloomberg).diff(), np.log(ambos.yahoo).diff()

tabla_fuentes = pd.DataFrame({
    "indicador": ["Ruedas en común", "Ruedas solo en Bloomberg", "Ruedas solo en Yahoo",
                  "Ruedas con diferencia de cierre > 0,1%", "Diferencia absoluta media del cierre (%)",
                  "Correlación de retornos diarios", "Desvío de retornos Bloomberg (%)", "Desvío de retornos Yahoo (%)"],
    "valor": [len(ambos), comp.yahoo.isna().sum(), comp.bloomberg.isna().sum(), int((dif.abs() > 0.001).sum()),
              round(dif.abs().mean() * 100, 4), round(ret_b.corr(ret_y), 5),
              round(ret_b.std() * 100, 4), round(ret_y.std() * 100, 4)]})
print(ambos[dif.abs() > 0.001].assign(dif_pct=(dif[dif.abs() > 0.001] * 100).round(2)))
tabla_fuentes

ERROR:yfinance:$^MERV: possibly delisted; no price data found  (1d 2020-11-01 -> 2026-03-31) (Yahoo error = "Data doesn't exist for startDate = 1604199600, endDate = 1774926000")


yfinance no disponible, se usa el CSV de la versión anterior: yfinance devolvió una serie vacía
            bloomberg      yahoo  dif_pct
fecha                                    
2023-05-03   288848.2   288008.0     0.29
2024-06-24  1559667.0  1554951.0     0.30
2024-07-02  1634019.0  1636391.0    -0.14
2024-07-31  1507785.0  1481853.0     1.75
2024-12-02  2299129.0  2295432.0     0.16
2025-10-13  1924930.0  1958155.0    -1.70


,indicador,valor
0,Ruedas en común,1220.00000
1,Ruedas solo en Bloomberg,2.00000
2,Ruedas solo en Yahoo,0.00000
3,"Ruedas con diferencia de cierre > 0,1%",6.00000
4,Diferencia absoluta media del cierre (%),0.00380
5,Correlación de retornos diarios,0.99932
6,Desvío de retornos Bloomberg (%),2.73010
7,Desvío de retornos Yahoo (%),2.72940


In [23]:
merval = bbg.copy()
merval.loc[pd.Timestamp("2025-10-13")] = yahoo.loc["2025-10-13"]                 # dato repetido en Bloomberg
merval = pd.concat([merval, yahoo.loc["2025-12-31":FIN_MUESTRA]]).sort_index()   # tramo que falta en Bloomberg
merval = merval[~merval.index.duplicated(keep="first")].loc[:FIN_MUESTRA]

ret = np.log(merval).diff().dropna().rename("ret_merval")
dias = ret.index
print("Merval final:", merval.index.min().date(), "a", merval.index.max().date(), "|", len(ret), "retornos")

Merval final: 2020-12-30 a 2026-02-27 | 1260 retornos


## Parte 3 — IPC observado y fecha real del evento

Igual que en la versión anterior: IPC Nivel General, región Nacional (INDEC), y fecha real de difusión de cada dato tomada de los 10 calendarios oficiales de INDEC.

In [24]:
url_ipc = "https://www.indec.gob.ar/ftp/cuadros/economia/serie_ipc_divisiones.csv"
try:
    ipc_raw = pd.read_csv(url_ipc, sep=";", encoding="latin-1", dtype=str)
    ipc = ipc_raw[(ipc_raw["Codigo"] == "0") & (ipc_raw["Region"] == "Nacional")].copy()
    ipc["fecha_dato"] = pd.to_datetime(ipc["Periodo"].astype(str) + "01", format="%Y%m%d")
    ipc["v_m_IPC"] = ipc["v_m_IPC"].str.replace(",", ".").astype(float)
    ipc = ipc[(ipc.fecha_dato >= "2021-01-01") & (ipc.fecha_dato <= "2025-12-01")][["fecha_dato", "v_m_IPC"]]
except Exception as e:
    print("INDEC no disponible, se usa la base anterior:", e)
    ipc = pd.read_csv(RUTA + "base_completa_sorpresas_merval.csv", parse_dates=["fecha_dato"])[["fecha_dato", "v_m_IPC"]]
ipc = ipc.sort_values("fecha_dato").reset_index(drop=True)

fechas_publicacion_ipc = {
    "2021-01-01": "2021-02-11", "2021-02-01": "2021-03-11", "2021-03-01": "2021-04-16",
    "2021-04-01": "2021-05-13", "2021-05-01": "2021-06-16", "2021-06-01": "2021-07-15",
    "2021-07-01": "2021-08-12", "2021-08-01": "2021-09-14", "2021-09-01": "2021-10-14",
    "2021-10-01": "2021-11-11", "2021-11-01": "2021-12-14", "2021-12-01": "2022-01-13",
    "2022-01-01": "2022-02-15", "2022-02-01": "2022-03-15", "2022-03-01": "2022-04-13",
    "2022-04-01": "2022-05-12", "2022-05-01": "2022-06-14", "2022-06-01": "2022-07-14",
    "2022-07-01": "2022-08-11", "2022-08-01": "2022-09-14", "2022-09-01": "2022-10-14",
    "2022-10-01": "2022-11-15", "2022-11-01": "2022-12-15", "2022-12-01": "2023-01-12",
    "2023-01-01": "2023-02-14", "2023-02-01": "2023-03-14", "2023-03-01": "2023-04-14",
    "2023-04-01": "2023-05-12", "2023-05-01": "2023-06-14", "2023-06-01": "2023-07-13",
    "2023-07-01": "2023-08-15", "2023-08-01": "2023-09-13", "2023-09-01": "2023-10-12",
    "2023-10-01": "2023-11-13", "2023-11-01": "2023-12-13", "2023-12-01": "2024-01-11",
    "2024-01-01": "2024-02-14", "2024-02-01": "2024-03-12", "2024-03-01": "2024-04-12",
    "2024-04-01": "2024-05-14", "2024-05-01": "2024-06-13", "2024-06-01": "2024-07-12",
    "2024-07-01": "2024-08-14", "2024-08-01": "2024-09-11", "2024-09-01": "2024-10-10",
    "2024-10-01": "2024-11-12", "2024-11-01": "2024-12-11", "2024-12-01": "2025-01-14",
    "2025-01-01": "2025-02-13", "2025-02-01": "2025-03-14", "2025-03-01": "2025-04-11",
    "2025-04-01": "2025-05-14", "2025-05-01": "2025-06-12", "2025-06-01": "2025-07-14",
    "2025-07-01": "2025-08-13", "2025-08-01": "2025-09-10", "2025-09-01": "2025-10-14",
    "2025-10-01": "2025-11-12", "2025-11-01": "2025-12-11", "2025-12-01": "2026-01-13",
}
ipc["fecha_publicacion_indec"] = pd.to_datetime(ipc.fecha_dato.dt.strftime("%Y-%m-%d").map(fechas_publicacion_ipc))
print(ipc.shape, "| fechas faltantes:", ipc.fecha_publicacion_indec.isna().sum())

(60, 3) | fechas faltantes: 0


## Parte 4 — REM: encuesta del mismo mes y encuesta del mes anterior

Para cada mes $t$ el REM pronostica la inflación de $t$ en varias encuestas sucesivas. Se toman dos:

- **Mismo mes (medida principal)**: la encuesta relevada a fines del propio mes $t$. Es la última expectativa disponible antes de que INDEC publique el dato, a mediados de $t+1$.
- **Mes anterior (robustez)**: la de fines de $t-1$, que era la medida de la versión anterior.

El ejemplo que motivó el cambio: en diciembre 2023 la encuesta de fines de noviembre esperaba 17,1% y la de fines de diciembre, ya con la devaluación del 13-dic, esperaba 27,0%. Con el dato observado de 25,5%, la sorpresa pasa de +8,4 a −1,5 p.p.

In [25]:
rem = pd.read_excel(RUTA + "historico-relevamiento-expectativas-mercado.xlsx", sheet_name="Base de Datos Completa", header=1)
rem = rem[(rem["Variable"] == "Precios minoristas (IPC nivel general; INDEC)") & (rem["Referencia"] == "var. % mensual")].copy()
rem["fecha_rem"] = pd.to_datetime(rem["Fecha de pronóstico"]); rem["periodo"] = pd.to_datetime(rem["Período"])

mismo = rem[(rem.fecha_rem.dt.to_period("M") == rem.periodo.dt.to_period("M"))]
mismo = mismo.sort_values("fecha_rem").groupby("periodo").last()[["fecha_rem", "Mediana"]]
mismo.columns = ["fecha_rem_mismo_mes", "rem_mismo_mes"]

anterior = rem[rem.periodo > rem.fecha_rem].sort_values("fecha_rem").groupby("periodo").last()[["fecha_rem", "Mediana"]]
anterior.columns = ["fecha_rem_mes_anterior", "rem_mes_anterior"]

ev = ipc.merge(mismo, left_on="fecha_dato", right_index=True, how="left") \
        .merge(anterior, left_on="fecha_dato", right_index=True, how="left")
ev["sorpresa"] = ev.v_m_IPC - ev.rem_mismo_mes
ev["sorpresa_ant"] = ev.v_m_IPC - ev.rem_mes_anterior
ev["abs_sorpresa"] = ev.sorpresa.abs(); ev["abs_sorpresa_ant"] = ev.sorpresa_ant.abs()
ev["dias_rem_a_ipc"] = (ev.fecha_publicacion_indec - ev.fecha_rem_mismo_mes).dt.days
ev["dias_rem_ant_a_ipc"] = (ev.fecha_publicacion_indec - ev.fecha_rem_mes_anterior).dt.days

assert (ev.fecha_rem_mismo_mes < ev.fecha_publicacion_indec).all(), "Hay una encuesta posterior al dato"
print(ev[["sorpresa", "sorpresa_ant", "dias_rem_a_ipc", "dias_rem_ant_a_ipc"]].describe().round(2))

       sorpresa  sorpresa_ant  dias_rem_a_ipc  dias_rem_ant_a_ipc
count     60.00         60.00           60.00               60.00
mean      -0.10          0.15           13.68               44.12
std        0.75          1.88            1.37                2.16
min       -2.60         -4.80           10.00               41.00
25%       -0.40         -0.50           13.00               42.00
50%        0.00          0.20           14.00               44.00
75%        0.40          0.71           14.00               46.00
max        1.20          8.40           18.00               49.00


## Parte 5 — Retornos anormales (modelo de media constante)

Para cada evento $e$, el retorno normal es la media de los retornos diarios en la ventana de estimación $[-120,-6]$ ruedas (se exige un mínimo de 15). El retorno anormal es $AR_{\tau}=r_{\tau}-\hat\mu_{e}$ y el CAR suma los $AR$ de la ventana del evento. El día 0 es la primera rueda en o después de la fecha de publicación (INDEC publica a las 16 h, antes del cierre de BYMA). También se guarda el CAR crudo, que era la medida de la versión anterior.

In [26]:
def pos_evento(f, idx):
    return idx.searchsorted(pd.Timestamp(f))

def car(r, f, a, b, media_constante=True, est=(-120, -6), minobs=15):
    p = pos_evento(f, r.index)
    suma = r.iloc[p + a: p + b + 1].sum()
    if not media_constante:
        return suma, np.nan
    estim = r.iloc[max(0, p + est[0]): p + est[1] + 1]
    if len(estim) < minobs:
        return np.nan, len(estim)
    return suma - (b - a + 1) * estim.mean(), len(estim)

VENTANAS = {"m1_p1": (-1, 1), "0_p1": (0, 1), "m1_0": (-1, 0), "m2_p2": (-2, 2)}
for k, (a, b) in VENTANAS.items():
    ev[f"car_{k}"] = ev.fecha_publicacion_indec.apply(lambda f: car(ret, f, a, b)[0])
    ev[f"car_raw_{k}"] = ev.fecha_publicacion_indec.apply(lambda f: car(ret, f, a, b, media_constante=False)[0])
ev["n_estimacion"] = ev.fecha_publicacion_indec.apply(lambda f: car(ret, f, -1, 1)[1])

# Merval en dólares (CCL GGAL)
ccl_d = ccl.reindex(merval.index, method="ffill")
ret_usd = np.log(merval / ccl_d).diff().dropna()
ev["car_usd_m1_p1"] = ev.fecha_publicacion_indec.apply(lambda f: car(ret_usd, f, -1, 1)[0])

# Acciones individuales (Bloomberg)
for t in TICKERS:
    rt = np.log(series[t]).diff().dropna().loc[:FIN_MUESTRA]
    ev[f"car_{t}"] = ev.fecha_publicacion_indec.apply(lambda f: car(rt, f, -1, 1)[0])

print("Ruedas de estimación (mín./media):", ev.n_estimacion.min(), round(ev.n_estimacion.mean(), 1))
print("Faltantes:", ev.filter(like="car_").isna().sum().sum())

Ruedas de estimación (mín./media): 23 110.7
Faltantes: 0


## Parte 6 — Contexto: VIX, EMBI y crisis cambiaria

- **VIX**: promedio en la ventana $[-1,+1]$ del evento (fuente: Yahoo Finance, `^VIX`).
- **EMBI**: riesgo país 5 ruedas antes del evento, para que no esté contaminado por la reacción al dato.
- **Crisis cambiaria**: un evento es de crisis si se publica dentro de los 45 días posteriores a un salto diario del dólar oficial BNA mayor al 10% (devaluación discreta). Como robustez se guardan ventanas de 30 y 60 días. Una regla con el CCL nominal no sirve: con inflación de dos dígitos marca como crisis buena parte de 2022-2023 solo por el arrastre inflacionario.

In [27]:
vix = series["vix"]
def media_ventana(s, f, a, b):
    p = pos_evento(f, dias)
    return s.loc[dias[p + a]: dias[p + b]].mean()
ev["vix_evento"] = ev.fecha_publicacion_indec.apply(lambda f: media_ventana(vix, f, -1, 1))

embi = leer_dos_columnas(RUTA + "EMBI 2020-2026.xlsx")
embi = embi[embi.index.weekday < 5]
ev["embi_pre"] = ev.fecha_publicacion_indec.apply(lambda f: embi.asof(dias[pos_evento(f, dias) - 5]))

ws = openpyxl.load_workbook(RUTA + "Dolar BNA.xlsx", data_only=True, read_only=True).worksheets[0]
bna = pd.Series({pd.Timestamp(r[1]): float(r[3]) for r in ws.iter_rows(values_only=True)
                 if r and isinstance(r[1], dt.datetime) and r[3] is not None}).sort_index()
saltos = bna.pct_change()
saltos = saltos[saltos > 0.10]
print("Saltos diarios del oficial > 10%:", {d.date(): round(v * 100, 1) for d, v in saltos.items()})

def en_crisis(f, ventana):
    f = pd.Timestamp(f)
    return int(any(0 <= (f - s).days <= ventana for s in saltos.index))
for d in (30, 45, 60):
    ev[f"crisis_{d}"] = ev.fecha_publicacion_indec.apply(lambda f: en_crisis(f, d))
ev["crisis"] = ev["crisis_45"]
print("Eventos en crisis (45 días):", ev.loc[ev.crisis == 1, "fecha_dato"].dt.strftime("%Y-%m").tolist())

QUIEBRE = pd.Timestamp("2023-12-10")
ev["post_milei"] = (ev.fecha_publicacion_indec >= QUIEBRE).astype(int)
ev["regimen"] = np.where(ev.post_milei == 1, "2024-2025 (post-Milei)", "2021-2023 (pre-Milei)")

Saltos diarios del oficial > 10%: {datetime.date(2023, 8, 14): 22.4, datetime.date(2023, 12, 13): 104.7, datetime.date(2025, 4, 14): 12.1}
Eventos en crisis (45 días): ['2023-07', '2023-08', '2023-11', '2023-12', '2025-04']


## Parte 7 — Exportación para R

In [28]:
diario = pd.DataFrame({"fecha": merval.index, "merval_close": merval.values})
diario["ret_merval"] = np.log(diario.merval_close).diff()
diario = diario.merge(yahoo.rename("merval_yahoo").rename_axis("fecha").reset_index(), on="fecha", how="left")

ev.to_csv("base_eventos_v2.csv", index=False)
diario.to_csv("merval_diario_v2.csv", index=False)
tabla_fuentes.to_csv("tabla_yahoo_vs_bloomberg.csv", index=False)
print(ev.shape, diario.shape)
print("Faltantes en la base de eventos:", int(ev.isna().sum().sum()))

(60, 38) (1261, 4)
Faltantes en la base de eventos: 0


**Próximo paso:** copiar los tres CSV a la carpeta de datos del proyecto en R y correr `Sorpresas_Merval_v2.Rmd`, que estima el AR(1)-GARCH(1,1)-t y todas las regresiones.